## Baseline Model Evaluation — Dummy vs Logistic Regression

### Understanding the metrics

**Precision** — Of all the customers the model predicted would churn, how many actually did? High precision means fewer wasted win-back offers on loyal customers.

**Recall** — Of all customers who actually churned, how many did the model catch? High recall means fewer missed at-risk customers slipping away unnoticed.

**Why PR-AUC over accuracy on imbalanced data?** Accuracy treats all errors equally and rewards models for getting the majority class right — with a 67%/33% split, a model that predicts "Active" for everyone gets 67% accuracy while being completely useless. PR-AUC summarizes precision/recall tradeoff across every threshold and focuses specifically on how well the model identifies the minority (churned) class, making it a much more honest measure of real performance here.

**What the Dummy Classifier tells us:** it establishes the floor. With 0% precision and 0% recall on the churned class (despite 67% accuracy), it proves that accuracy alone would be a misleading metric for this problem, and gives a PR-AUC baseline (0.33, ≈ the churn rate) that any real model must beat.

---

### Incident: Data leakage in the initial Logistic Regression run

**What happened:** the first run of the Logistic Regression baseline produced suspiciously perfect results — 99.6% precision, 100% recall, and PR-AUC of 1.0.Real-world churn models rarely exceed ~80-85% PR-AUC, so this was a red flag rather than a success.

**Why it happened:** the `Churned` label is defined directly from `LastPurchaseDate` (no purchase in the 90 days before the snapshot date = churned). The `Recency` feature also computed directly from `LastPurchaseDate` (days since snapshot). Because both were derived from the same underlying field with the same cutoff logic, `Recency` was almost a direct proxy for the label itself — the model wasn't learning customer behavior, it was just decoding the churn label's own definition.

**How it was solved:** removed `Recency` from the model's feature set. It's still present in the underlying data for reference/display purposes (e.g. useful to show "last purchase was X days ago" in the dashboard later), but excluded from training to prevent the model from directly seeing a near-restatement of the target.

**Why this solution was chosen over alternatives:** an alternative was to bucket `Recency` into coarse ranges (e.g. <30 / 30-60 / 60-90+ days) rather than dropping it entirely, which would keep some signal while reducing the exact leakage. This was rejected for a portfolio project where clarity and defensibility matter more than marginal performance gains — a fully dropped feature is simple to explain and leaves no ambiguity about whether leakage remains. Dropping it entirely was thesafer, more interview-defensible choice.

**Result after the fix:**

Dummy Classifier: precision 0.00, recall 0.00, F1 0.00, ROC-AUC 0.50, PR-AUC 0.33

Logistic Regression: precision 0.54, recall 0.81, F1 0.65, ROC-AUC 0.80, PR-AUC 0.62

Logistic Regression beats the dummy baseline across every metric, most notably 
PR-AUC (0.33 → 0.62), which is the one that matters most given the class imbalance.


This is a believable, healthy baseline: a meaningful lift over the dummy classifier (PR-AUC 0.33 → 0.62) driven by genuine behavioral signal (Frequency, Monetary, AvgOrderValue, etc.), with a sensible precision/recall tradeoff — high recall (81%) at some cost to precision (54%), appropriate for a use case where missing a real churner is more costly than a wasted win-back offer. This tradeoff will be tuned more precisely in Day 5's business cost analysis.

**Goal for Day 3:** XGBoost should beat this Logistic Regression baseline (PR-AUC 0.62) on F1 and PR-AUC.